<a href="https://colab.research.google.com/github/GuyilhermePedrosa/INF-102-TRABALHOS-2026-I_Guilherme-Osorio-Pedrosa/blob/main/INF102_Cpp_MySQL_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:

import subprocess, time

print('Atualizando lista de pacotes...')
r = subprocess.run(
    ['apt-get', 'update', '--fix-missing', '-qq'],
    capture_output=True, text=True
)
print('OK' if r.returncode == 0 else r.stderr[:300])

print('Instalando mysql-server e libmysqlcppconn-dev...')
r = subprocess.run(
    ['apt-get', 'install', '-y', '--fix-missing',
     'mysql-server', 'libmysqlcppconn-dev'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(' Dependências instaladas!')
else:
    print(' Erro na instalação:')
    print(r.stderr[:500])

Atualizando lista de pacotes...
OK
Instalando mysql-server e libmysqlcppconn-dev...
 Dependências instaladas!


In [50]:

import subprocess, time

print('Iniciando MySQL...')
subprocess.run(['service', 'mysql', 'start'], capture_output=True)
time.sleep(5)
./
r = subprocess.run(['service', 'mysql', 'status'], capture_output=True, text=True)
saida = r.stdout + r.stderr

if 'active' in saida or 'running' in saida.lower():
    print(' MySQL rodando!')
else:
    print(' MySQL pode não ter iniciado. Saída:')
    print(saida[:300])

Iniciando MySQL...
 MySQL pode não ter iniciado. Saída:
 * /usr/bin/mysqladmin  Ver 8.0.46-0ubuntu0.22.04.3 for Linux on x86_64 ((Ubuntu))
Copyright (c) 2000, 2026, Oracle and/or its affiliates.

Oracle is a registered trademark of Oracle Corporation and/or its
affiliates. Other names may be trademarks of their respective
owners.

Server version		8.0.46-


In [51]:

import subprocess

setup_sql = """
CREATE DATABASE IF NOT EXISTS empresa;
USE empresa;
CREATE TABLE IF NOT EXISTS funcionario (
    matricula INT PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    salario DOUBLE NOT NULL
);
CREATE USER IF NOT EXISTS 'aluno'@'localhost' IDENTIFIED BY 'senha123';
GRANT ALL PRIVILEGES ON empresa.* TO 'aluno'@'localhost';
FLUSH PRIVILEGES;
"""

r = subprocess.run(
    ['mysql', '-u', 'root', '--execute', setup_sql],
    capture_output=True, text=True
)

if r.returncode == 0:
    print('Banco empresa e tabela funcionario criados!')
else:
    print('Erro:', r.stderr[:400])

Banco empresa e tabela funcionario criados!


In [52]:

import subprocess

cpp_code = r"""
#include <iostream>
#include <memory>
#include <mysql_driver.h>
#include <mysql_connection.h>
#include <cppconn/driver.h>
#include <cppconn/connection.h>
#include <cppconn/statement.h>
#include <cppconn/prepared_statement.h>
#include <cppconn/resultset.h>

using namespace std;

class Funcionario {
private:
    int matricula;
    string nome;
    double salario;
public:
    Funcionario() : matricula(0), nome(""), salario(0) {}
    Funcionario(int matricula, string nome, double salario)
        : matricula(matricula), nome(nome), salario(salario) {}
    int getMatricula() { return matricula; }
    string getNome()   { return nome; }
    double getSalario(){ return salario; }
    void setMatricula(int m) { matricula = m; }
    void setNome(string n)   { nome = n; }
    void setSalario(double s){ salario = s; }
};

class FuncionarioDAO {
private:
    sql::mysql::MySQL_Driver *driver;
    unique_ptr<sql::Connection> con;
public:
    FuncionarioDAO(string servidor, string usuario, string senha, string banco) {
        driver = sql::mysql::get_mysql_driver_instance();
        con.reset(driver->connect(servidor, usuario, senha));
        con->setSchema(banco);
    }
    void inserir(Funcionario f) {
        unique_ptr<sql::PreparedStatement> pstmt(
            con->prepareStatement(
                "INSERT INTO funcionario (matricula,nome,salario) VALUES (?,?,?)"));
        pstmt->setInt(1, f.getMatricula());
        pstmt->setString(2, f.getNome());
        pstmt->setDouble(3, f.getSalario());
        pstmt->execute();
        cout << "\nFuncionario inserido com sucesso!\n";
    }
    void listar() {
        unique_ptr<sql::Statement> stmt(con->createStatement());
        unique_ptr<sql::ResultSet> res(
            stmt->executeQuery("SELECT * FROM funcionario ORDER BY matricula"));
        cout << "\n=== FUNCIONARIOS ===\n";
        bool vazio = true;
        while (res->next()) {
            vazio = false;
            cout << "Matricula: " << res->getInt("matricula") << "\n"
                 << "Nome:      " << res->getString("nome") << "\n"
                 << "Salario:   R$ " << res->getDouble("salario") << "\n"
                 << "--------------------\n";
        }
        if (vazio) cout << "Nenhum funcionario cadastrado.\n";
    }
    void atualizarSalario(int matricula, double salario) {
        unique_ptr<sql::PreparedStatement> pstmt(
            con->prepareStatement(
                "UPDATE funcionario SET salario=? WHERE matricula=?"));
        pstmt->setDouble(1, salario);
        pstmt->setInt(2, matricula);
        int linhas = pstmt->executeUpdate();
        cout << (linhas > 0 ? "\nSalario atualizado.\n" : "\nFuncionario nao encontrado.\n");
    }
    void excluir(int matricula) {
        unique_ptr<sql::PreparedStatement> pstmt(
            con->prepareStatement(
                "DELETE FROM funcionario WHERE matricula=?"));
        pstmt->setInt(1, matricula);
        int linhas = pstmt->executeUpdate();
        cout << (linhas > 0 ? "\nFuncionario removido.\n" : "\nFuncionario nao encontrado.\n");
    }
    Funcionario buscar(int matricula) {
        unique_ptr<sql::PreparedStatement> pstmt(
            con->prepareStatement(
                "SELECT * FROM funcionario WHERE matricula=?"));
        pstmt->setInt(1, matricula);
        unique_ptr<sql::ResultSet> res(pstmt->executeQuery());
        if (res->next())
            return Funcionario(res->getInt("matricula"),
                               res->getString("nome"),
                               res->getDouble("salario"));
        return Funcionario();
    }
};

int main() {
    try {
        FuncionarioDAO dao("tcp://127.0.0.1:3306", "aluno", "senha123", "empresa");
        int opcao;
        do {
            cout << "\n========== MENU ==========\n";
            cout << "1 - Inserir Funcionario\n";
            cout << "2 - Listar Funcionarios\n";
            cout << "3 - Buscar Funcionario\n";
            cout << "4 - Atualizar Salario\n";
            cout << "5 - Excluir Funcionario\n";
            cout << "0 - Sair\n";
            cout << "Opcao: ";
            cin >> opcao;
            switch (opcao) {
                case 1: {
                    int mat; string nome; double sal;
                    cout << "Matricula: "; cin >> mat;
                    cin.ignore();
                    cout << "Nome: "; getline(cin, nome);
                    cout << "Salario: "; cin >> sal;
                    dao.inserir(Funcionario(mat, nome, sal));
                    break;
                }
                case 2: dao.listar(); break;
                case 3: {
                    int mat;
                    cout << "Matricula: "; cin >> mat;
                    Funcionario f = dao.buscar(mat);
                    if (f.getMatricula() != 0)
                        cout << "\nMatricula: " << f.getMatricula()
                             << "\nNome: " << f.getNome()
                             << "\nSalario: R$ " << f.getSalario() << "\n";
                    else
                        cout << "\nFuncionario nao encontrado.\n";
                    break;
                }
                case 4: {
                    int mat; double sal;
                    cout << "Matricula: "; cin >> mat;
                    cout << "Novo salario: "; cin >> sal;
                    dao.atualizarSalario(mat, sal);
                    break;
                }
                case 5: {
                    int mat;
                    cout << "Matricula: "; cin >> mat;
                    dao.excluir(mat);
                    break;
                }
                case 0: cout << "\nEncerrando...\n"; break;
                default: cout << "\nOpcao invalida.\n";
            }
        } while (opcao != 0);
    } catch (sql::SQLException &e) {
        cerr << "\nErro MySQL: " << e.what() << endl;
    }
    return 0;
}
"""

with open('/content/funcionario.cpp', 'w') as f:
    f.write(cpp_code)
print('Arquivo funcionario.cpp criado!')

r = subprocess.run(
    ['g++', '/content/funcionario.cpp', '-o', '/content/funcionario', '-lmysqlcppconn'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(' Compilado com sucesso!')
else:
    print(' Erro de compilação:')
    print(r.stderr)

Arquivo funcionario.cpp criado!
 Compilado com sucesso!


In [53]:

import subprocess

print('Executando CRUD: inserir 3, listar, buscar, atualizar, excluir, listar final...')
print()

entrada = (
    '1\n101\nAna Silva\n3500.00\n'
    '1\n102\nCarlos Souza\n4200.50\n'
    '1\n103\nMaria Oliveira\n5100.75\n'
    '2\n'
    '3\n102\n'
    '4\n101\n6000\n'
    '5\n103\n'
    '2\n'
    '0\n'
)

r = subprocess.run(
    ['/content/funcionario'],
    input=entrada,
    capture_output=True,
    text=True
)

print(r.stdout)
if r.stderr:
    print('ERROS:', r.stderr[:400])

print()
print('=== Estado final no banco MySQL ===')
r2 = subprocess.run(
    ['mysql', '-u', 'aluno', '-psenha123', 'empresa',
     '--execute', 'SELECT * FROM funcionario ORDER BY matricula;'],
    capture_output=True, text=True
)
print(r2.stdout)
if r2.stderr:
    print(r2.stderr[:200])

print(' Concluído!')

Executando CRUD: inserir 3, listar, buscar, atualizar, excluir, listar final...


========== MENU ==========
1 - Inserir Funcionario
2 - Listar Funcionarios
3 - Buscar Funcionario
4 - Atualizar Salario
5 - Excluir Funcionario
0 - Sair
Opcao: Matricula: Nome: Salario: 
ERROS: 
Erro MySQL: Duplicate entry '101' for key 'funcionario.PRIMARY'


=== Estado final no banco MySQL ===
matricula	nome	salario
101	Ana Silva	6000
102	Carlos Souza	4200.5
27822	guilherme	17100

mysql: [Warning] Using a password on the command line interface can be insecure.

 Concluído!
